In [ ]:
import os
from dotenv import load_dotenv
from langchain_core.prompts import PromptTemplate
from langchain_docling.loader import ExportType

load_dotenv()

# https://github.com/huggingface/transformers/issues/5486:
os.environ["TOKENIZERS_PARALLELISM"] = "false"

FILE_PATH = ["29500_i30.pdf"]  # Docling Technical Report
EXPORT_TYPE = ExportType.DOC_CHUNKS

TOP_K = 3

In [ ]:
api_key= os.getenv("OPEN_AI_KEY")

In [ ]:
from docling_core.transforms.chunker.tokenizer.huggingface import HuggingFaceTokenizer
from transformers import AutoTokenizer

from docling.chunking import HybridChunker
from sentence_transformers import SentenceTransformer

# Load the model
#model = SentenceTransformer("Qwen/Qwen3-Embedding-0.6B")
EMBED_MODEL_ID = "Qwen/Qwen3-Embedding-0.6B"
MAX_TOKENS = 64

tokenizer = HuggingFaceTokenizer(
    tokenizer=AutoTokenizer.from_pretrained(EMBED_MODEL_ID),
    max_tokens=MAX_TOKENS
)

In [ ]:
from langchain_docling import DoclingLoader

from docling.chunking import HybridChunker

chunker = HybridChunker(
    tokenizer=tokenizer,
    merge_peers=True,  # optional, defaults to True
)



#Chunk grouping in docling format?  
loader = DoclingLoader(
    file_path=FILE_PATH,
    export_type=EXPORT_TYPE,
    chunker=chunker,
)

docs = loader.load()



In [ ]:
type(docs)

In [ ]:
import pickle

with open('my_list.pkl', 'wb') as f:
    pickle.dump(docs, f)


In [ ]:
with open("my_list.pkl", 'rb') as f:
    p = pickle.load(f)

len(p)

In [ ]:
if EXPORT_TYPE == ExportType.DOC_CHUNKS:
    splits = docs

elif EXPORT_TYPE == ExportType.MARKDOWN:
    from langchain_text_splitters import MarkdownHeaderTextSplitter

    splitter = MarkdownHeaderTextSplitter(
        headers_to_split_on=[
            ("#", "Header_1"),
            ("##", "Header_2"),
            ("###", "Header_3"),
        ],
    )
    splits = [split for doc in docs for split in splitter.split_text(doc.page_content)]
else:
    raise ValueError(f"Unexpected export type: {EXPORT_TYPE}")
    

In [ ]:
len(splits)

In [ ]:
for d in splits[:3]:
    print(f"- {d.page_content=}")
print("...")

In [ ]:
import json
from pathlib import Path
from tempfile import mkdtemp
from langchain_huggingface.embeddings import HuggingFaceEmbeddings
from langchain_chroma import Chroma
from uuid import uuid4
from langchain_community.vectorstores.utils import filter_complex_metadata 
embedding = HuggingFaceEmbeddings(model_name=EMBED_MODEL_ID)


vector_store = Chroma(
    collection_name="example_collection",
    embedding_function=embedding,
    persist_directory="./chroma_langchain_db",  # Where to save data locally, remove if not necessary
)
uuids = [str(uuid4()) for _ in range(len(splits_list))]

filter_complex_metadata(splits_list)

vector_store.add_documents(documents=splits_list, ids=uuids)

In [ ]:
from langchain.chains import create_retrieval_chain
from langchain.chains.combine_documents import create_stuff_documents_chain
from langchain_openai import OpenAI

retriever = vector_store.as_retriever(search_kwargs={"k": TOP_K})
llm = OpenAI(api_key=api_key)


def clip_text(text, threshold=100):
    return f"{text[:threshold]}..." if len(text) > threshold else text

In [ ]:
retr = retriever.invoke("Who is the author of this Document?")

texts= [(d.page_content, d.metadata) for d in retr]

print(texts)

In [ ]:
from langchain_openai import OpenAI

llm = OpenAI(api_key=api_key)

prompt = PromptTemplate.from_template("You are currently a part of a RAG. Using these files: {input}, answer the user's prompt: {prompt} with the at most precesion possible. Keep your answers clear and with as little filler as possible. Be to the point! If you believe that the piece of information within the prompt is not found in the given document chunks, say so rather than making up information. At the begining of your answer, cite the prompt given to you.")

chain = prompt | llm
print(chain.invoke(
    {
        "prompt": "Who is the author of this Document?",
        "input": texts
    }
))